# Prediction model walkthrough

Run this notebook from the `prediction-server` directory using its virtual environment. It shows the complete flow: input records, validation, feature engineering, CatBoost training, evaluation, prediction output, and retraining after data updates.

This notebook trains only from the local development database. It does not generate or use synthetic training data.

## 1. Load training input

Each record contains the formula, machine/mold, process settings, optional environment measurements, and observed lab-result metrics. Eligible records are completed/scored runs and testing runs that already have summarized test results.

In [ ]:
from pathlib import Path
from pprint import pprint

from datetime import date
from urllib.parse import quote

from prediction_server.domain.features import (
    categorical_columns, feature_columns, flatten_input, matrix, validate_consistent_types,
)
from prediction_server.domain.prediction_service import PredictionService
from prediction_server.domain.schemas import PredictionInput, TrainRequest
from prediction_server.infrastructure.current_schema_repository import CurrentSchemaRepository
from prediction_server.infrastructure.model_registry import FileModelRegistry

def load_env(path):
    return dict(line.strip().split('=', 1) for line in Path(path).read_text().splitlines() if '=' in line and not line.lstrip().startswith('#'))

db = load_env('../main-server/.env.development')
database_url = f"postgresql://{quote(db['DB_USER'])}:{quote(db['DB_PASSWORD'])}@{db['DB_HOST']}:{db['DB_PORT']}/{db['DB_NAME']}"
real_data_repository = CurrentSchemaRepository(database_url)
real_records = real_data_repository.load_training_records()
training_request = TrainRequest(dataset_version=f'amfpi-real-{date.today().isoformat()}', records=real_records, random_seed=42)
print(f'Dataset: {training_request.dataset_version}')
print(f'Training records: {len(training_request.records)}')
pprint(training_request.records[0].model_dump(mode='json'))

## 2. Validate and engineer features

The service validates that composition totals 100%, each process value has exactly one numeric or text value, and that a feature does not switch between numeric and text types. It then flattens each nested record into model columns.

In [ ]:
feature_rows = [flatten_input(record) for record in training_request.records]
columns = feature_columns(feature_rows)
validate_consistent_types(feature_rows, columns)
categorical = categorical_columns(feature_rows, columns)
values = matrix(feature_rows, columns, categorical)

print(f'Feature count: {len(columns)}')
print('Categorical features:')
pprint(categorical)
print('\nFirst flattened input row:')
pprint(feature_rows[0])
print('\nFirst model matrix row:')
pprint(dict(zip(columns, values[0])))

## 3. Train and validate the CatBoost models

The service creates one `CatBoostRegressor` for each outcome shared by every record. It evaluates using a deterministic 80/20 split, then retrains each final saved model on all records. Current settings are: 300 iterations, depth 6, learning rate 0.05, and RMSE loss.

In [ ]:
model_directory = Path('.models')
service = PredictionService(FileModelRegistry(model_directory))
train_result = service.train(training_request)

print(f'Model ID: {train_result.model_id}')
print(f'Predicted metrics: {train_result.target_metrics}')
print('Validation results:')
for metric, evaluation in train_result.validation.items():
    print(f'  {metric}: MAE={evaluation.mae}, R²={evaluation.r2}')

## 4. Use a production-style input and inspect the output

A prediction uses formula/process input only. It does not include the actual output metrics, because those are what the model estimates.

In [ ]:
prediction_input = training_request.records[0].model_copy(deep=True)
prediction_input.outcomes = {}  # Outcomes are unknown at prediction time.
prediction = service.predict(train_result.model_id, prediction_input)

print('Prediction input:')
pprint(prediction_input.model_dump(mode='json', exclude={'outcomes', 'production_run_id'}))
print('\nPrediction output:')
pprint(prediction)

## 5. Test a new formula

Edit only the percentage values in `new_composition` below. Keep the total at 100. Use material codes that already appear in the real training data; a material with no training history cannot be predicted reliably. The batch/machine/process fields are copied from a real run—edit them when those measured values are available.

In [ ]:
# Start with an actual formula so material IDs, supplier IDs, and equipment IDs are valid.
new_formula_data = training_request.records[0].model_dump(
    mode='json', exclude={'outcomes', 'production_run_id'}
)

# EDIT THESE VALUES. They must add up to 100.
new_composition = {
    component['material_code']: component['percentage']
    for component in new_formula_data['formula']['components']
}
# Example for a two-material formula:
# new_composition = {'YOUR-RESIN-CODE': 70.0, 'YOUR-FILLER-CODE': 30.0}

if abs(sum(new_composition.values()) - 100.0) > 0.01:
    raise ValueError(f'Formula percentages must total 100; received {sum(new_composition.values()):g}')

for component in new_formula_data['formula']['components']:
    component['percentage'] = new_composition[component['material_code']]

new_formula_input = PredictionInput.model_validate(new_formula_data)
new_formula_prediction = service.predict(train_result.model_id, new_formula_input)
print('Formula being tested:')
pprint(new_composition)
print('\nPredicted lab properties:')
pprint(new_formula_prediction)

## 6. Refresh real data and retrain

When newly tested production runs are summarized, run the next cell to reload them from the database and create a new real-data model. Every retraining operation produces a new model ID; update `PREDICTION_MODEL_ID` in `main-server/.env.development` to promote that model for dashboard use.

In [ ]:
updated_request = TrainRequest(
    dataset_version=f'amfpi-real-{date.today().isoformat()}',
    records=real_data_repository.load_training_records(),
    random_seed=42,
)
updated_result = service.train(updated_request)
print(f'Updated model ID: {updated_result.model_id}')
print('Updated validation:')
pprint({metric: evaluation.model_dump() for metric, evaluation in updated_result.validation.items()})

## 7. Inspect the real database data

This read-only section lists the real, populated properties and measurement outputs in the local development database.

In [ ]:
import html
from IPython.display import HTML, display
import psycopg

def env_values(path):
    return dict(line.strip().split('=', 1) for line in Path(path).read_text().splitlines() if '=' in line and not line.lstrip().startswith('#'))

def table(title, rows):
    print(title)
    if not rows:
        display(HTML('<em>No populated values.</em>'))
        return
    names = rows[0].keys()
    head = ''.join(f'<th>{html.escape(name)}</th>' for name in names)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(str(row[name] if row[name] is not None else "—"))}</td>' for name in names) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

env = env_values('../main-server/.env.development')
with psycopg.connect(host=env['DB_HOST'], port=env['DB_PORT'], dbname=env['DB_NAME'], user=env['DB_USER'], password=env['DB_PASSWORD'], row_factory=psycopg.rows.dict_row) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT md.display_name AS output, md.metric_key, md.default_unit AS unit, count(DISTINCT s.production_run_id)::int AS completed_runs FROM metric_definitions md JOIN run_metric_summaries s ON s.metric_id = md.id GROUP BY md.id, md.display_name, md.metric_key, md.default_unit ORDER BY completed_runs DESC, md.display_name")
        table('Available model outputs', cursor.fetchall())
        cursor.execute("SELECT count(*)::int AS runs, count(DISTINCT formulation_id)::int AS formulations, count(DISTINCT machine_id)::int AS machines, count(DISTINCT mold_id)::int AS molds, count(DISTINCT injection_pressure)::int AS injection_pressure_values, count(DISTINCT melt_temperature)::int AS melt_temperature_values, count(DISTINCT cycle_time)::int AS cycle_time_values, count(DISTINCT cure_hours_before_test)::int AS cure_time_values FROM production_runs WHERE status IN ('testing', 'completed', 'scored')")
        table('Batch-level input variation', cursor.fetchall())
        cursor.execute("SELECT d.display_name AS process_property, d.parameter_key, d.default_unit AS unit, count(DISTINCT v.production_run_id)::int AS populated_runs FROM process_parameter_definitions d LEFT JOIN production_run_process_values v ON v.parameter_definition_id=d.id GROUP BY d.id, d.display_name, d.parameter_key, d.default_unit HAVING count(DISTINCT v.production_run_id)>0 ORDER BY populated_runs DESC, d.display_name")
        table('Detailed process properties captured by production run', cursor.fetchall())
        cursor.execute("SELECT d.canonical_name AS material_property, d.property_key, d.common_units AS units, count(f.id)::int AS populated_facts FROM material_property_definitions d JOIN material_property_facts f ON f.property_definition_id=d.id GROUP BY d.id, d.canonical_name, d.property_key, d.common_units ORDER BY populated_facts DESC, d.canonical_name LIMIT 20")
        table('Most populated material properties', cursor.fetchall())

print('A property with one distinct value cannot teach the model its effect. Record actual process values for future batches.')

## 8. Start the API and use the promoted model

Start the processor in a separate terminal:

```powershell
uvicorn prediction_server.main:app --host 127.0.0.1 --port 4100
```

After setting the new `PREDICTION_MODEL_ID` and restarting the main server, the dashboard sends the same production-run input to `POST /v1/predictions?model_id=<model-id>`.